<h1>Important</h1>

- The following notebook has only personal learning purposes with no further intention. This was developed using AI tools combined with multiple iterations to refine the code given at first it does generate many errors, from documentation error and more.
- The key intention of this notebook is to show to use a model available from Hugging Face in combination with techniques from the same library to do fine-tuning of the model and show how it works.
- This is not a comercial or industry code to be used, it is just a personal academic learning of how to use different python libraries with ideas of Reinforcement Learning and LLMs.

Key considerations when running this:
- The machine that was used had installed cuda nvidia with 8 GB of capacity, and the idea was to constraint the dataset size and memory usage when training the model.
- Do not load more than 1 model if the CUDA capacity is small because it will lead to potential crushing.
- Try to use cuda and not cpu because is much faster when running.

_____

# Supervised Fine-Tuning (SFT)

In [1]:
# Supervised Fine-Tuning (SFT) - Standalone Implementation
# Mathematical Foundation: L_SFT = -∑ log P(y_t | x, y_<t; θ)

import warnings

warnings.filterwarnings("ignore")

import subprocess
import sys
import torch
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
import json
from datetime import datetime

# SFT Configuration
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TEMPERATURE = 0.8
MAX_LENGTH = 512
MAX_NEW_TOKENS = 200
LEARNING_RATE_SFT = 4e-5
NUM_EPOCHS_SFT = 20
BATCH_SIZE_SFT = 1
GRAD_ACCUM_SFT = 16
WARMUP_RATIO = 0.2
LOGGING_STEPS = 5

device = "cuda" if torch.cuda.is_available() else "cpu"

TEST_QUESTIONS = [
    "How do I cook pasta perfectly?",
    "What's the best way to scramble eggs?",
    "How do I make rice that isn't sticky?",
    "What's an easy dinner for beginners?",
    "How do I know when chicken is cooked?",
]


def install_packages():
    packages = [
        "torch>=2.0.0",
        "transformers>=4.36.0",
        "trl>=0.7.4",
        "datasets>=2.14.0",
        "accelerate>=0.21.0",
    ]
    for pkg in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        except:
            pass


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def monitor_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(device) / (1024**3)
        reserved = torch.cuda.memory_reserved(device) / (1024**3)
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
        return reserved < 7.5
    return True


def test_model(model, tokenizer, prompt):
    model.eval()
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    )
    if torch.cuda.is_available():
        inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            use_cache=False,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return response.strip()


def evaluate_stage(model, tokenizer, stage_name):
    results = {}
    print(f"\n{'='*60}")
    print(f"{stage_name.upper()} MODEL EVALUATION")
    print(f"{'='*60}")

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(f"\nQ{i}: {question}")
        print("-" * 40)
        response = test_model(model, tokenizer, question)
        results[question] = response
        print(f"Answer: {response}")

    return results


def create_sft_dataset():
    """Create comprehensive SFT dataset for cooking domain"""
    examples = [
        {
            "text": "How do I cook pasta perfectly?\n\nCooking perfect pasta requires attention to water, salt, and timing.\n\n**Water Setup:**\nUse a large pot with plenty of water - about 4-6 quarts for 1 pound of pasta. The pasta needs room to move freely without sticking together.\n\n**Salt the Water:**\nAdd 1-2 tablespoons of salt when the water reaches a rolling boil. The water should taste like mild seawater. This is your only chance to season the pasta itself.\n\n**Cooking Process:**\nAdd pasta to boiling water and stir immediately to prevent sticking. Cook for 1-2 minutes less than package directions for al dente texture.\n\n**Testing Doneness:**\nTaste test frequently in the final minutes. Perfect pasta should have a slight firmness when you bite it - tender but not mushy.\n\n**Finishing:**\nReserve 1 cup of pasta water before draining. This starchy water helps bind sauces to pasta. Never rinse pasta unless making a cold salad."
        },
        {
            "text": "What's the best way to scramble eggs?\n\nGreat scrambled eggs are all about gentle heat and constant movement.\n\n**Preparation:**\nCrack 2-3 eggs per person into a bowl. Add a pinch of salt and 1 tablespoon of butter or cream per 2 eggs. Whisk until completely smooth with no streaks.\n\n**Heat Control:**\nUse medium-low heat in a non-stick pan. Add a small amount of butter and let it melt completely without browning.\n\n**Cooking Technique:**\nPour in eggs and let them sit for 20-30 seconds. Using a spatula, gently push cooked edges toward center, tilting pan to let raw egg flow underneath.\n\n**The Key:**\nContinuous gentle movement is crucial. Keep pushing and folding, never scrambling vigorously. Remove from heat when eggs still look slightly wet - they'll finish cooking from residual heat.\n\n**Serving:**\nServe immediately on warmed plates. Season with fresh pepper and herbs if desired."
        },
        {
            "text": "How do I make rice that isn't sticky?\n\nPerfect rice comes down to the right ratio, technique, and not lifting the lid.\n\n**Rice-to-Water Ratio:**\nFor long-grain white rice, use 1 cup rice to 1.5 cups water. For brown rice, use 1 cup rice to 2 cups water.\n\n**Rinsing:**\nRinse rice in cold water until water runs clear - usually 3-4 rinses. This removes excess starch that causes stickiness.\n\n**Cooking Method:**\nCombine rice and water in heavy-bottomed pot. Bring to a boil, then immediately reduce to lowest heat setting. Cover tightly with lid.\n\n**Timing:**\nWhite rice: 18 minutes. Brown rice: 45 minutes. Never lift the lid during cooking - this releases steam needed for proper cooking.\n\n**Resting:**\nAfter cooking time, remove from heat but keep lid on for 10 minutes. Then fluff gently with fork, not spoon.\n\n**Pro Tip:**\nFor extra fluffy rice, add 1 teaspoon of butter or oil to the water before cooking."
        },
        {
            "text": "What's an easy dinner for beginners?\n\nOne-pan chicken and vegetables is perfect for beginners - minimal cleanup and hard to mess up.\n\n**Ingredients:**\n4 chicken thighs, 2 cups baby potatoes (halved), 1 cup carrots (chopped), 1 onion (sliced), olive oil, salt, pepper, dried herbs.\n\n**Preparation:**\nPreheat oven to 425°F. Pat chicken dry and season generously with salt and pepper. Cut vegetables into similar-sized pieces so they cook evenly.\n\n**Assembly:**\nToss vegetables with 2 tablespoons olive oil, salt, and pepper. Spread on large baking sheet. Place seasoned chicken on top, skin side up.\n\n**Cooking:**\nBake for 35-40 minutes until chicken reaches 165°F internal temperature and vegetables are tender when pierced with fork.\n\n**Serving:**\nLet rest 5 minutes before serving. The chicken juices flavor the vegetables beautifully.\n\n**Variations:**\nTry different vegetable combinations - broccoli, bell peppers, zucchini all work well."
        },
        {
            "text": "How do I know when chicken is cooked?\n\nFood safety is crucial with chicken - use multiple indicators to ensure doneness.\n\n**Internal Temperature:**\nUse instant-read thermometer in thickest part of meat, not touching bone. Chicken must reach 165°F (74°C) throughout.\n\n**Visual Cues:**\nProperly cooked chicken has no pink color in the meat. Juices should run clear, not pink or red, when pierced with knife.\n\n**Texture Test:**\nCooked chicken feels firm to touch, not soft or squishy. Raw chicken has a distinctly different, softer texture.\n\n**Cooking Times:**\nBoneless breasts: 6-8 minutes per side. Bone-in thighs: 25-30 minutes total. Whole chicken: 20 minutes per pound at 375°F.\n\n**Rest Time:**\nLet chicken rest 5-10 minutes after cooking. Internal temperature will continue rising 5-10 degrees, ensuring safety.\n\n**Safety Note:**\nWhen in doubt, cook longer. Overcooked chicken is better than foodborne illness. Clean all surfaces that touched raw chicken."
        },
    ]
    return examples


def run_sft_training(model, tokenizer):
    print("=" * 80)
    print("SUPERVISED FINE-TUNING (SFT)")
    print("=" * 80)
    print("Mathematical Foundation: L_SFT = -∑ log P(y_t | x, y_<t; θ)")
    print(f"Training: {NUM_EPOCHS_SFT} epochs, LR: {LEARNING_RATE_SFT}")

    sft_data = create_sft_dataset()
    dataset = Dataset.from_list(sft_data)

    print(f"Dataset: {len(sft_data)} comprehensive cooking examples")
    print(f"Effective batch size: {BATCH_SIZE_SFT * GRAD_ACCUM_SFT}")

    try:
        from trl import SFTTrainer, SFTConfig

        config = SFTConfig(
            output_dir="./temp_sft",
            num_train_epochs=NUM_EPOCHS_SFT,
            per_device_train_batch_size=BATCH_SIZE_SFT,
            gradient_accumulation_steps=GRAD_ACCUM_SFT,
            learning_rate=LEARNING_RATE_SFT,
            max_length=MAX_LENGTH,
            logging_steps=LOGGING_STEPS,
            save_strategy="no",
            fp16=False,
            bf16=torch.cuda.is_available(),
            warmup_ratio=WARMUP_RATIO,
            remove_unused_columns=False,
            dataset_text_field="text",
            gradient_checkpointing=True,
            dataloader_drop_last=True,
        )

        trainer = SFTTrainer(
            model=model,
            args=config,
            train_dataset=dataset,
            processing_class=tokenizer,
        )

        trainer.train()
        del trainer

    except Exception as e:
        print(f"SFT training error: {e}")

    cleanup_memory()
    return model, tokenizer


def save_results_json(results, filename):
    os.makedirs("./results", exist_ok=True)
    with open(f"./results/{filename}", "w") as f:
        json.dump(
            {
                "timestamp": datetime.now().isoformat(),
                "model_outputs": results,
                "test_questions": TEST_QUESTIONS,
            },
            f,
            indent=2,
        )


def main():
    print("=" * 80)
    print("SUPERVISED FINE-TUNING (SFT) - STANDALONE")
    print("=" * 80)
    print("Foundation method for LLM task adaptation")
    print("Mathematical basis: Cross-entropy loss on next token prediction")
    print("=" * 80)

    install_packages()

    print(f"\nInitializing model: {MODEL_NAME}")

    # Load model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )

    # Configure tokenizer
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model.resize_token_embeddings(len(tokenizer))
    monitor_memory()

    # Evaluate base model
    print("\n" + "=" * 50)
    print("EVALUATING BASE MODEL")
    print("=" * 50)
    base_results = evaluate_stage(model, tokenizer, "BASE")
    save_results_json(base_results, "sft_base_results.json")

    # Run SFT training
    print("\n" + "=" * 50)
    print("STARTING SFT TRAINING")
    print("=" * 50)
    model, tokenizer = run_sft_training(model, tokenizer)

    # Evaluate trained model
    print("\n" + "=" * 50)
    print("EVALUATING TRAINED MODEL")
    print("=" * 50)
    trained_results = evaluate_stage(model, tokenizer, "SFT")
    save_results_json(trained_results, "sft_trained_results.json")

    # Save final model
    print(f"\nSaving SFT-trained model...")
    os.makedirs("./models/sft_standalone", exist_ok=True)
    model.save_pretrained("./models/sft_standalone")
    tokenizer.save_pretrained("./models/sft_standalone")

    # Compare results
    print(f"\n{'='*80}")
    print("COMPARATIVE ANALYSIS: Base vs SFT")
    print(f"{'='*80}")

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(f"\n[QUESTION {i}]: {question}")
        print("=" * 60)
        print(f"\n[BASE MODEL]:")
        print(f"{base_results[question]}")
        print(f"\n[SFT MODEL]:")
        print(f"{trained_results[question]}")
        print("=" * 60)

    print(f"\n{'='*80}")
    print("SFT TRAINING COMPLETED")
    print(f"{'='*80}")
    print("Results saved to:")
    print("  • ./models/sft_standalone/ - Trained model")
    print("  • ./results/sft_base_results.json - Base evaluation")
    print("  • ./results/sft_trained_results.json - Trained evaluation")
    print(f"{'='*80}")

In [2]:
# Execute main
if __name__ == "__main__":
    main()

SUPERVISED FINE-TUNING (SFT) - STANDALONE
Foundation method for LLM task adaptation
Mathematical basis: Cross-entropy loss on next token prediction

Initializing model: Qwen/Qwen2.5-0.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


GPU Memory: 0.92GB allocated, 1.19GB reserved

EVALUATING BASE MODEL

BASE MODEL EVALUATION

Q1: How do I cook pasta perfectly?
----------------------------------------
Answer: Cooking pasta to perfection can be achieved through careful preparation, the use of appropriate cooking techniques, and proper storage. Here are some tips on how to cook pasta to perfection:

1. Choose the right type of pasta: There are several types of pasta, each with its own texture and flavor. For best results, choose a type that is easy to work with and has a consistent texture.

2. Prepare the pasta: Rinse the pasta under cold water to remove any excess starch and then drain it well. Toss the pasta with olive oil, salt, and pepper before draining again.

3. Cook the pasta: Heat a large pot or saucepan over medium-high heat. Add a pinch of salt and cook the pasta according to package instructions until al dente (about 8-10 minutes). Drain the pasta and set aside in a colander to cool completely.

4. Adjust 

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,1.720200
10,0.109500
15,0.007100
20,0.005600



EVALUATING TRAINED MODEL

SFT MODEL EVALUATION

Q1: How do I cook pasta perfectly?
----------------------------------------
Answer: Cooking perfect pasta requires attention to water, salt, and timing.

**Water Setup:**
Use a large pot with plenty of water - about 4-6 quarts for 1 pound of pasta. The pasta needs room to move freely without sticking together.

**Salt the Water:**
Add 1-2 tablespoons of salt when the water reaches a rolling boil. The water should taste like mild seawater. This is your only chance to season the pasta itself.

**Cooking Process:**
Add pasta to boiling water and stir immediately to prevent sticking. Cook for 1-2 minutes less than package directions for al dente texture.

**Testing Doneness:**
Taste test frequently in the final minutes. Perfect pasta should have a slight firmness when you bite it - tender but not mushy.

**Finishing:**
Reserve 1 cup of pasta water before draining. This starchy water helps bind sauces to pasta. Never rinse pasta unless making